In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score ## 評估指標函式庫

In [ ]:
# ==========================================
# 區塊 1：硬體與超參數設定
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"目前訓練使用的硬體: {device}")

# 訓練參數設定
HIDDEN_DIM = 16
OUTPUT_DIM = 2
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001

# ==========================================
# 區塊 2：資料載入與嚴謹預處理 (已修復 CUDA 報錯問題)
# ==========================================
# 1. 讀取資料
test_data = pd.read_csv('./data_set/balance_data/all_balance_data.csv')

# 2. 標籤轉換 (3轉1, 2轉0)
test_data['DIQ010'] = test_data['DIQ010'].replace({3.0: 1.0, 2.0: 0.0})

# 4. 切分特徵與標籤
X = test_data.drop(columns=['SEQN', 'DIQ010'])
Y = test_data['DIQ010']

# 讓程式自動抓取真實的特徵數量
INPUT_DIM = X.shape[1] 

# 5. 拆分訓練與測試集
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

# 6. 特徵縮放
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 區塊 3：轉為 Tensor 並封裝 DataLoader
# ==========================================
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long) 

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=BATCH_SIZE, shuffle=False)

print(f"資料準備完畢！特徵數: {INPUT_DIM}, 訓練集: {len(X_train_tensor)}筆, 測試集: {len(X_test_tensor)}筆")

# ==========================================
# 區塊 4：模型定義
# ==========================================
class ChronicDiseaseModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(ChronicDiseaseModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        # 注意：不加 Softmax，因為 CrossEntropyLoss 內建了
        
    def forward(self, x):
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out)
        return out

# 實例化模型，並搬移到 GPU
model = ChronicDiseaseModel(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM).to(device)

# ==========================================
# 區塊 5：裁判與教練
# ==========================================
criterion = nn.CrossEntropyLoss() 
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# ==========================================
# 區塊 6：訓練迴圈 (Training Loop)
# ==========================================
print("\n--- 開始在 GPU 上訓練模型 ---")
for epoch in range(EPOCHS):
    model.train() # 切換為訓練模式
    epoch_loss = 0.0
    
    for batch_X, batch_y in train_loader:
        # 將資料搬移到 GPU
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()               # 清空梯度
        outputs = model(batch_X)            # 前向傳播 (打分數)
        loss = criterion(outputs, batch_y)  # 裁判計算 Loss
        loss.backward()                     # 反向傳播 (算微積分)
        optimizer.step()                    # 教練更新權重
        
        epoch_loss += loss.item()
        
    # 每 10 個 Epoch 印出一次訓練進度
    if (epoch + 1) % 10 == 0:
        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{EPOCHS}], 平均 Loss: {avg_loss:.4f}")

# ==========================================
# 區塊 7：模型測試與評估
# ==========================================
print("\n---模型測試與最終評估 ---")
model.eval() # 宣告模型進入「測試模式」

all_preds = []
all_targets = []

with torch.no_grad(): # 關閉微積分計算，節省顯存
    for batch_X, batch_y in test_loader:
        batch_X = batch_X.to(device)
        
        outputs = model(batch_X)
        _, predicted = torch.max(outputs.data, 1)
        
        # 收集預測結果並搬回 CPU
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(batch_y.numpy())

# 印出最終報告

cm = confusion_matrix(y_test, all_preds)
print("--- 混淆矩陣 (Confusion Matrix) ---")
print(cm)

target_names = ['Health (0)', 'Risk (1)']
print(classification_report(all_targets, all_preds, target_names=target_names, zero_division=0))

🚀 目前訓練使用的硬體: cuda
✅ 資料準備完畢！特徵數: 13, 訓練集: 5828筆, 測試集: 1458筆

--- 🏃 開始在 GPU 上訓練模型 ---
Epoch [10/50], 平均 Loss: 0.3324
Epoch [20/50], 平均 Loss: 0.3282
Epoch [30/50], 平均 Loss: 0.3245
Epoch [40/50], 平均 Loss: 0.3206
Epoch [50/50], 平均 Loss: 0.3213

--- 📊 模型測試與最終評估 ---


NameError: name 'confusion_matrix' is not defined